In [11]:
from dotenv import load_dotenv
load_dotenv()

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
loader = PyPDFLoader("../data/data_science_syllabus.pdf")

docs = loader.load()


splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitted_docs = splitter.split_documents(docs)

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vectorstore = Chroma.from_documents(documents=splitted_docs, embedding=embeddings)


llm = ChatOpenAI(model="gpt-5")

def get_context(query):
    data = vectorstore.similarity_search(query)
    context = ""
    for doc in data:
        context += doc.page_content + "\n"
    return {"context": context, "question": query}

prompt = PromptTemplate.from_template(
    "You are a helpful assistant for answering questions about the Data Science Syllabus. If you don't know the answer, just say that you don't know, don't try to make up an answer. Use the following context to answer the question.\n\nContext:\n{context}\n\nQuestion: {question}\n"
)


rag_chain= get_context | prompt | llm


response = rag_chain.invoke("what is the duration module 3?")
print(response.content)

3 weeks (it runs in Month 3).
